# Tutorial 05 — Créer un nouveau modèle non linéaire pairwise : Lotka-Volterra

Ce tutorial montre pas à pas comment ajouter un **nouveau modèle non linéaire pairwise** dans `awesomepkf`,
en prenant l'exemple du modèle proie-prédateur de **Lotka-Volterra** discrétisé à l'ordre 1.

**Ce que vous allez apprendre :**

1. Comprendre la structure d'un modèle pairwise (`BaseModelGxGy`)
2. Générer les fichiers Python du modèle pairwise et de sa version augmentée
3. Vérifier que le modèle est automatiquement découvert par `ModelFactoryNonLinear`
4. Explorer le modèle (paramètres, équations LaTeX, jacobiens symboliques)
5. Simuler une trajectoire et visualiser le portrait de phase
6. Appliquer les filtres EPKF, UPKF, PPF et PF sur ce modèle

**Prérequis :** Tutorial 02 — Nonlinear Models

## Setup

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from prg import (
    NonLinear_EPKF,
    NonLinear_UPKF,
    NonLinear_PPF,
    NonLinear_PF,
    ParamNonLinear,
    ModelFactoryNonLinear,
    __version__,
)

print(f"awesomepkf version: {__version__}")

SEED = 42
N    = 200

---
## 1. Le modèle de Lotka-Volterra pairwise

### Dynamique continue

Le modèle de Lotka-Volterra décrit l'évolution de populations de **proies** ($x_1$) et de **prédateurs** ($x_2$) :

$$\dot{x}_1 = \alpha\, x_1 - \beta\, x_1 x_2, \qquad \dot{x}_2 = \delta\, x_1 x_2 - \gamma\, x_2$$

| Paramètre | Rôle | Valeur choisie |
|-----------|------|---------------|
| $\alpha$ | Taux de croissance des proies | 0.5 |
| $\beta$ | Taux de prédation | 0.1 |
| $\gamma$ | Taux de mortalité des prédateurs | 0.4 |
| $\delta$ | Efficacité de conversion | 0.05 |

**Point d'équilibre non trivial :** $(x_1^*, x_2^*) = (\gamma/\delta,\; \alpha/\beta) = (8,\; 5)$

---
### Discrétisation Euler explicite (ordre 1), pas $\Delta t = 0.3$

$$x_1^{k+1} = (1 + \alpha\Delta t)\,x_1^k - \beta\Delta t\,x_1^k x_2^k$$
$$x_2^{k+1} = (1 - \gamma\Delta t)\,x_2^k + \delta\Delta t\,x_1^k x_2^k$$

---
### Formulation pairwise (`BaseModelGxGy`)

Dans la formulation pairwise, $g_x$ **et** $g_y$ peuvent simultanément dépendre
de l'état courant $\mathbf{x}$ **et** de l'observation précédente $\mathbf{y}$.
On ajoute ici un couplage $\eta$ (rétroaction $y \to$ dynamique via $\tanh$)
et un couplage croisé $c$ (entre les deux observations).

**Transition** $g_x(\mathbf{x}, \mathbf{y})$ :
$$g_{x_1} = (1 + \alpha\Delta t)\,x_1 - \beta\Delta t\,x_1 x_2 + \eta\,\tanh(y_2) + v^x_1$$
$$g_{x_2} = (1 - \gamma\Delta t)\,x_2 + \delta\Delta t\,x_1 x_2 - \eta\,\tanh(y_1) + v^x_2$$

**Observation** $g_y(\mathbf{x}, \mathbf{y})$ :
$$g_{y_1} = x_1 + c\,y_2 + v^y_1, \qquad g_{y_2} = x_2 - c\,y_1 + v^y_2$$

avec $\eta = 0.05$ et $c = 0.20$.

> **Note :** les jacobiens $A_n = \partial g / \partial \mathbf{z}$ et $B_n = \partial g / \partial \mathbf{v}$
> sont **calculés automatiquement par SymPy** dans `BaseModelGxGy` — il suffit de définir `symbolic_model()`.

---
## 2. Générer le fichier du modèle pairwise

La cellule suivante génère le fichier `model_x2_y2_LotkaVolterra_pairwise.py`.

> **Action requise :** copiez (ou déplacez) ce fichier dans :
> ```
> prg/models/nonLinear/
> ```
> La factory `ModelFactoryNonLinear` scanne ce répertoire automatiquement au démarrage
> — aucune modification du code existant n'est nécessaire.
>
> La cellule ci-dessous écrit directement dans le bon répertoire si vous exécutez le notebook
> depuis `ipynb/` (chemin relatif `../prg/models/nonLinear/`).

In [ ]:
pairwise_code = '''\
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import numpy as np
import sympy as sp

from prg.models.nonLinear.base_model_gxgy import BaseModelGxGy
from prg.utils.exceptions import NumericalError

__all__ = ["Model_x2_y2_LotkaVolterra_pairwise"]


class Model_x2_y2_LotkaVolterra_pairwise(BaseModelGxGy):
    """
    Modele proie-predateur de Lotka-Volterra discretise a l ordre 1
    (Euler explicite), sous forme pairwise (dim_x=2, dim_y=2).

    Parametres biologiques :
        ALPHA = 0.5   taux de croissance des proies
        BETA  = 0.1   taux de predation
        GAMMA = 0.4   taux de mortalite des predateurs
        DELTA = 0.05  efficacite de conversion predation -> croissance
        DT    = 0.3   pas de discretisation temporelle

    Couplage pairwise :
        ETA = 0.05    retroaction observation -> dynamique (via tanh)
        C   = 0.20    couplage croise observation -> observation

    Equation de transition gx (depend de x et y) :
        gx1 = (1 + ALPHA*DT)*x1 - BETA*DT*x1*x2 + ETA*tanh(y2) + t1
        gx2 = (1 - GAMMA*DT)*x2 + DELTA*DT*x1*x2 - ETA*tanh(y1) + t2

    Equation d observation gy (depend de x et y) :
        gy1 = x1 + C*y2 + u1
        gy2 = x2 - C*y1 + u2

    Les jacobiens An = dg/dz et Bn = dg/dn sont calcules automatiquement
    par SymPy dans BaseModelGxGy.

    Point d equilibre non trivial : (x1*, x2*) = (GAMMA/DELTA, ALPHA/BETA) = (8, 5)
    """

    ALPHA: float = 0.5
    BETA:  float = 0.1
    GAMMA: float = 0.4
    DELTA: float = 0.05
    DT:    float = 0.3
    ETA:   float = 0.05
    C:     float = 0.20

    def __init__(self):
        # Les attributs de classe sont accessibles avant super().__init__()
        # (super() appelle _build_symbolic_model() qui appelle symbolic_model())
        super().__init__(dim_x=2, dim_y=2, model_type="nonlinear")

        # Point d equilibre non trivial
        x1_eq = self.GAMMA / self.DELTA   # 8.0
        x2_eq = self.ALPHA / self.BETA    # 5.0

        try:
            self.mQ, self.mz0, self.Pz0 = self._init_random_params(
                self.dim_x, self.dim_y, 0.10, seed=None
            )
            # On demarre autour du point d equilibre
            self.mz0 = np.array([[x1_eq], [x2_eq], [x1_eq], [x2_eq]])
        except (ValueError, np.exceptions.AxisError) as e:
            raise NumericalError(
                f"[{self.MODEL_NAME}] Initialization failed: {e}"
            ) from e

    # ------------------------------------------------------------------
    def symbolic_model(self, sx, sy, st, su):
        x1, x2 = sx[0], sx[1]
        y1, y2 = sy[0], sy[1]
        t1, t2 = st[0], st[1]
        u1, u2 = su[0], su[1]

        sgx = sp.Matrix([
            (1 + self.ALPHA * self.DT) * x1
            - self.BETA  * self.DT * x1 * x2
            + self.ETA   * sp.tanh(y2) + t1,
            (1 - self.GAMMA * self.DT) * x2
            + self.DELTA * self.DT * x1 * x2
            - self.ETA   * sp.tanh(y1) + t2,
        ])

        sgy = sp.Matrix([
            x1 + self.C * y2 + u1,
            x2 - self.C * y1 + u2,
        ])

        return sgx, sgy
'''

target_pairwise = Path("../prg/models/nonLinear/model_x2_y2_LotkaVolterra_pairwise.py")
target_pairwise.write_text(pairwise_code, encoding="utf-8")
print(f"Fichier cree : {target_pairwise.resolve()}")

---
## 3. Générer le fichier du modèle augmenté

La version **augmentée** réinterprète l'état joint pairwise $\mathbf{z} = (x_1, x_2, y_1, y_2)$
comme un **état augmenté** de dimension 4, avec une observation identité $h(\mathbf{x}_{aug}) = (x_3, x_4)$.

Elle hérite de `BaseModelFxHx` et réutilise les expressions symboliques du modèle pairwise
pour construire la dynamique augmentée $f(\mathbf{x}_{aug})$.

> **Action requise :** idem — le fichier est écrit dans `prg/models/nonLinear/`.

In [ ]:
augmented_code = '''\
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import numpy as np
import sympy as sp

from prg.models.nonLinear.base_model_fxhx import BaseModelFxHx
from prg.models.nonLinear.model_x2_y2_LotkaVolterra_pairwise import (
    Model_x2_y2_LotkaVolterra_pairwise,
)
from prg.utils.exceptions import NumericalError

__all__ = ["Model_x2_y2_LotkaVolterra_augmented"]


class Model_x2_y2_LotkaVolterra_augmented(BaseModelFxHx):
    """
    Version augmentee de Model_x2_y2_LotkaVolterra_pairwise (BaseModelFxHx).

    L etat pairwise z = (x1, x2, y1, y2) est reinterprete comme etat augmente
    x_aug = (x1, x2, y1, y2) avec dim_x = 4, dim_y = 2.

    La dynamique f(x_aug) reprend les equations gx et gy du modele pairwise.
    L observation h(x_aug) = (x3, x4) = (y1, y2) est une identite pure.
    """

    def __init__(self):
        self.mod = Model_x2_y2_LotkaVolterra_pairwise()

        dim_x  = self.mod.dim_x    # 2
        dim_y  = self.mod.dim_y    # 2
        dim_xy = self.mod.dim_xy   # 4

        super().__init__(
            dim_x=dim_x + dim_y,   # 4
            dim_y=dim_y,           # 2
            model_type="nonlinear",
            augmented=True,
        )

        try:
            self.mQ = np.zeros((dim_xy + dim_y, dim_xy + dim_y))
            self.mQ[0:dim_xy, 0:dim_xy] = self.mod.mQ

            self.mz0 = np.zeros((dim_xy + dim_y, 1))
            self.mz0[0:dim_xy] = self.mod.mz0
            self.mz0[dim_xy : dim_xy + dim_y] = self.mz0[dim_xy - dim_y : dim_xy]

            self.Pz0 = np.zeros((dim_xy + dim_y, dim_xy + dim_y))
            self.Pz0[0:dim_xy, 0:dim_xy] = self.mod.Pz0
            self.Pz0[dim_xy : dim_xy + dim_y, :] = self.Pz0[dim_xy - dim_y : dim_xy, :]
            self.Pz0[:, dim_xy : dim_xy + dim_y] = self.Pz0[:, dim_xy - dim_y : dim_xy]

        except (ValueError, IndexError) as e:
            raise NumericalError(
                f"[{self.MODEL_NAME}] Initialization failed: {e}"
            ) from e

    # ------------------------------------------------------------------
    def symbolic_model(self, sx, st, su):
        """
        sx : sp.Matrix(4, 1) -> [x1, x2, y1, y2]  (etat augmente)
        st : sp.Matrix(4, 1) -> [t0, t1, t2, t3]  (bruits process)
        su : sp.Matrix(2, 1) -> [u0, u1]           (bruits observation, non utilises dans h)
        """
        mx0 = sp.Symbol("x0", real=True)
        mx1 = sp.Symbol("x1", real=True)
        my0 = sp.Symbol("y0", real=True)
        my1 = sp.Symbol("y1", real=True)
        mt0 = sp.Symbol("t0", real=True)
        mt1 = sp.Symbol("t1", real=True)
        mu0 = sp.Symbol("u0", real=True)
        mu1 = sp.Symbol("u1", real=True)

        subs_state = {mx0: sx[0], mx1: sx[1], my0: sx[2], my1: sx[3]}

        sfx = sp.Matrix([
            self.mod._sgx.subs({**subs_state, mt0: st[0]})[0],  # gx1
            self.mod._sgx.subs({**subs_state, mt1: st[1]})[1],  # gx2
            self.mod._sgy.subs({**subs_state, mu0: st[2]})[0],  # gy1
            self.mod._sgy.subs({**subs_state, mu1: st[3]})[1],  # gy2
        ])

        # h(x_aug) = [x3, x4] = [y1, y2] -- identite pure
        shx = sp.Matrix([[sx[2]], [sx[3]]])

        # -- Diagnostic ----------------------------------------------------
        expected_fx = set(sx.free_symbols) | set(st.free_symbols)
        expected_hx = set(sx.free_symbols) | set(su.free_symbols)
        residual_fx = sfx.free_symbols - expected_fx
        residual_hx = shx.free_symbols - expected_hx

        if residual_fx:
            print(f"[WARN] sfx contient des symboles non attendus : {residual_fx}")
        if residual_hx:
            print(f"[WARN] shx contient des symboles non attendus : {residual_hx}")

        return sfx, shx
'''

target_augmented = Path("../prg/models/nonLinear/model_x2_y2_LotkaVolterra_augmented.py")
target_augmented.write_text(augmented_code, encoding="utf-8")
print(f"Fichier cree : {target_augmented.resolve()}")

---
## 4. Vérifier l'enregistrement dans la factory

`ModelFactoryNonLinear` scanne automatiquement le répertoire `prg/models/nonLinear/`
et enregistre toute classe concrète héritant de `BaseModelNonLinear`.
Le **nom de modèle** est dérivé du nom de classe : `Model_x2_y2_LotkaVolterra_pairwise`
→ `model_x2_y2_LotkaVolterra_pairwise`.

> Si le noyau Python est déjà démarré depuis avant la création des fichiers,
> rechargez-le (`Kernel > Restart`) pour que la factory redécouvre les nouveaux modules.

In [ ]:
# Forcer la redécouverte (utile si la factory a déjà scanné avant l'ajout des fichiers)
ModelFactoryNonLinear._registry.clear()

models = sorted(ModelFactoryNonLinear.list_models())
print(f"{len(models)} modeles non lineaires disponibles:")
for m in models:
    marker = "  <<< NOUVEAU" if "LotkaVolterra" in m else ""
    print(f"  {m}{marker}")

---
## 5. Explorer le modèle

### 5.1 Paramètres et informations de base

In [ ]:
model_lv = ModelFactoryNonLinear.create("model_x2_y2_LotkaVolterra_pairwise")
params   = model_lv.get_params()

print(f"Modele   : {model_lv}")
print(f"dim_x    : {params['dim_x']}  |  dim_y : {params['dim_y']}")
print(f"Pairwise : {params['pairwiseModel']}")
print()
print(f"Point d equilibre : x1* = {model_lv.GAMMA/model_lv.DELTA:.1f},  x2* = {model_lv.ALPHA/model_lv.BETA:.1f}")
print()
print("mz0 (etat initial moyen) :", params['mz0'].flatten())
print()
print("mQ (matrice de covariance du bruit joint, shape", params['mQ'].shape, "):")
print(np.round(params['mQ'], 4))

### 5.2 Équations symboliques et jacobiens (générés automatiquement par SymPy)

In [ ]:
from IPython.display import display, Math

latex_str = model_lv.latex_model()
display(Math(latex_str))

In [ ]:
import sympy as sp

print("=== gx (transition) ===")
sp.pprint(model_lv._sgx, use_unicode=True)
print()
print("=== gy (observation) ===")
sp.pprint(model_lv._sgy, use_unicode=True)
print()
print("=== An = dg/dz (jacobien d etat, calcule par SymPy) ===")
sp.pprint(model_lv._sAn, use_unicode=True)
print()
print("=== Bn = dg/dnoise ===")
sp.pprint(model_lv._sBn, use_unicode=True)

---
## 6. Portrait de phase — comportement sans bruit

On simule le système **déterministe** (sans bruit) pour visualiser les orbites
caractéristiques du modèle de Lotka-Volterra.
Plusieurs conditions initiales autour du point d'équilibre $(8, 5)$ sont tracées.

In [ ]:
def simulate_lv_deterministic(x1_0, x2_0, alpha, beta, gamma, delta, dt, n_steps):
    """Simule le systeme de Lotka-Volterra discret sans bruit."""
    x1, x2 = x1_0, x2_0
    traj = [(x1, x2)]
    for _ in range(n_steps):
        x1_new = (1 + alpha * dt) * x1 - beta * dt * x1 * x2
        x2_new = (1 - gamma * dt) * x2 + delta * dt * x1 * x2
        x1, x2 = x1_new, x2_new
        traj.append((x1, x2))
    return np.array(traj)

alpha, beta, gamma, delta, dt = (
    model_lv.ALPHA, model_lv.BETA, model_lv.GAMMA, model_lv.DELTA, model_lv.DT
)
x1_eq, x2_eq = gamma / delta, alpha / beta   # (8.0, 5.0)

# Plusieurs orbites de rayon croissant autour de l equilibre
init_conditions = [
    (x1_eq + 2, x2_eq),
    (x1_eq + 4, x2_eq),
    (x1_eq + 6, x2_eq),
    (x1_eq,     x2_eq + 2),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Portrait de phase
ax_phase = axes[0]
colors_orb = ["C0", "C1", "C2", "C3"]
for (x10, x20), c in zip(init_conditions, colors_orb):
    traj = simulate_lv_deterministic(x10, x20, alpha, beta, gamma, delta, dt, n_steps=500)
    ax_phase.plot(traj[:, 0], traj[:, 1], color=c, lw=1.0, alpha=0.8)
    ax_phase.plot(x10, x20, "o", color=c, ms=5)

ax_phase.plot(x1_eq, x2_eq, "k*", ms=12, label=f"Equilibre ({x1_eq:.0f}, {x2_eq:.0f})")
ax_phase.set_xlabel(r"$x_1$ (proies)");  ax_phase.set_ylabel(r"$x_2$ (predateurs)")
ax_phase.set_title("Portrait de phase — orbites de Lotka-Volterra")
ax_phase.legend();  ax_phase.grid(True, ls="--", alpha=0.5)

# Serie temporelle (une orbite)
ax_time = axes[1]
traj_ref = simulate_lv_deterministic(x1_eq + 4, x2_eq, alpha, beta, gamma, delta, dt, n_steps=300)
t_det = np.arange(len(traj_ref)) * dt
ax_time.plot(t_det, traj_ref[:, 0], color="C0", lw=1.5, label=r"$x_1$ proies")
ax_time.plot(t_det, traj_ref[:, 1], color="C1", lw=1.5, label=r"$x_2$ predateurs")
ax_time.axhline(x1_eq, color="C0", ls=":", alpha=0.6, label=f"$x_1^*={x1_eq:.0f}$")
ax_time.axhline(x2_eq, color="C1", ls=":", alpha=0.6, label=f"$x_2^*={x2_eq:.0f}$")
ax_time.set_xlabel("Temps");  ax_time.set_ylabel("Population")
ax_time.set_title("Serie temporelle — systeme deterministe")
ax_time.legend(fontsize=8);  ax_time.grid(True, ls="--", alpha=0.5)

plt.tight_layout()
plt.show()

---
## 7. Trajectoire stochastique et observations

On simule maintenant le modèle **avec bruit** via `process_N_data()`
et on visualise l'état vrai et les observations bruitées.

In [ ]:
def make_param(model):
    """Construit un ParamNonLinear a partir d un modele."""
    p = model.get_params().copy()
    dx = p.pop("dim_x")
    dy = p.pop("dim_y")
    return ParamNonLinear(0, dx, dy, **p)

def extract(results):
    """Extrait (x_true, x_update) depuis les resultats d un filtre."""
    xu = np.array([r[4].flatten() for r in results if r[4] is not None])
    M  = len(xu)
    xt = np.array([r[1].flatten() for r in results[:M]])
    return xt, xu

def mse(xt, xu):
    return float(np.mean((xt - xu) ** 2))

param_lv = make_param(model_lv)

# Simulation via le filtre (on recupere etat vrai + observations)
epkf_sim = NonLinear_EPKF(param_lv, sKey=SEED)
sim_data  = epkf_sim.simulate_N_data(N)

# Extraction etat vrai et observations
x_true = np.array([r[1].flatten() for r in sim_data])  # (N+1, 2)
y_obs  = np.array([r[2].flatten() for r in sim_data])  # (N+1, 2)
t_sim  = np.arange(len(x_true))

fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)

labels_x = [r"$x_1$ (proies)",     r"$x_2$ (predateurs)"]
labels_y = [r"$y_1$ (obs. proies)", r"$y_2$ (obs. predateurs)"]
colors_x = ["C0", "C1"]

for i in range(2):
    # Etat vrai
    ax = axes[0, i]
    ax.plot(t_sim, x_true[:, i], color=colors_x[i], lw=1.4, label=labels_x[i])
    ax.axhline([x1_eq, x2_eq][i], color=colors_x[i], ls=":", alpha=0.5)
    ax.set_ylabel(labels_x[i]); ax.legend(fontsize=8); ax.grid(True, ls="--", alpha=0.5)
    ax.set_title(f"Etat vrai — {labels_x[i]}")

    # Observation
    ax = axes[1, i]
    ax.plot(t_sim, y_obs[:, i], color=colors_x[i], lw=0.8, alpha=0.7, label=labels_y[i])
    ax.plot(t_sim, x_true[:, i], color="black", lw=1.0, ls="--", alpha=0.5, label="Vrai")
    ax.set_xlabel("Pas de temps $k$"); ax.set_ylabel(labels_y[i])
    ax.legend(fontsize=8); ax.grid(True, ls="--", alpha=0.5)
    ax.set_title(f"Observation bruitee — {labels_y[i]}")

fig.suptitle("Trajectoire stochastique — modele Lotka-Volterra pairwise", y=1.01)
plt.tight_layout()
plt.show()

---
## 8. Portrait de phase stochastique

Le portrait de phase avec bruit illustre comment les orbites
se déforment par rapport au cas déterministe.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

# Orbite stochastique
ax.plot(x_true[:, 0], x_true[:, 1], color="C0", lw=1.0, alpha=0.8, label="Etat vrai (stochastique)")
ax.scatter(x_true[0, 0], x_true[0, 1], color="C0", s=80, zorder=5, label="Point initial")

# Orbite deterministe de reference (meme condition initiale)
traj_det = simulate_lv_deterministic(
    x_true[0, 0], x_true[0, 1], alpha, beta, gamma, delta, dt, n_steps=N
)
ax.plot(traj_det[:, 0], traj_det[:, 1], color="gray", lw=1.0, ls="--", alpha=0.7, label="Deterministe (reference)")

ax.plot(x1_eq, x2_eq, "k*", ms=12, label=f"Equilibre ({x1_eq:.0f}, {x2_eq:.0f})")
ax.set_xlabel(r"$x_1$ (proies)"); ax.set_ylabel(r"$x_2$ (predateurs)")
ax.set_title("Portrait de phase — stochastique vs deterministe")
ax.legend(fontsize=9); ax.grid(True, ls="--", alpha=0.5)
plt.tight_layout()
plt.show()

---
## 9. Filtrage — EPKF, UPKF, PPF, PF

On applique les quatre filtres disponibles sur la **même trajectoire** simulée.
Le modèle pairwise permet d'utiliser le **PPF** (Pairwise Particle Filter),
qui exploite la structure de rétroaction $y \to x$.

In [ ]:
# EPKF
epkf = NonLinear_EPKF(param_lv, sKey=SEED)
res_epkf = epkf.process_N_data(N=None, data_generator=iter(sim_data))
xt_epkf, xu_epkf = extract(res_epkf)
print(f"EPKF                   MSE = {mse(xt_epkf, xu_epkf):.6f}")

# UPKF
upkf = NonLinear_UPKF(param_lv, sigmaSet="wan2000", sKey=SEED)
res_upkf = upkf.process_N_data(N=None, data_generator=iter(sim_data))
xt_upkf, xu_upkf = extract(res_upkf)
print(f"UPKF (wan2000)         MSE = {mse(xt_upkf, xu_upkf):.6f}")

# PPF (pairwise uniquement)
ppf = NonLinear_PPF(param_lv, n_particles=500, sKey=SEED)
res_ppf = ppf.process_N_data(N=None, data_generator=iter(sim_data))
xt_ppf, xu_ppf = extract(res_ppf)
print(f"PPF  (500 particules)  MSE = {mse(xt_ppf, xu_ppf):.6f}")

# PF
pf = NonLinear_PF(param_lv, n_particles=500, sKey=SEED)
res_pf = pf.process_N_data(N=None, data_generator=iter(sim_data))
xt_pf, xu_pf = extract(res_pf)
print(f"PF   (500 particules)  MSE = {mse(xt_pf, xu_pf):.6f}")

### 9.1 Visualisation des estimées par composante

In [ ]:
M   = min(len(xu_epkf), len(xu_upkf), len(xu_ppf), len(xu_pf))
t   = np.arange(M)
WIN = slice(0, min(M, 100))   # zoom sur les 100 premiers pas

filter_results = {
    "EPKF":        (xt_epkf[:M], xu_epkf[:M], "C0", "-"),
    "UPKF (wan)": (xt_upkf[:M], xu_upkf[:M], "C1", "--"),
    "PPF": (xt_ppf[:M], xu_ppf[:M], "C2", "-."),
    "PF":         (xt_pf[:M],  xu_pf[:M],  "C3", ":"),
}

comp_labels = [r"$x_1$ (proies)", r"$x_2$ (predateurs)"]

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

for i, (ax, lbl) in enumerate(zip(axes, comp_labels)):
    ax.plot(t[WIN], xt_epkf[WIN, i], color="black", lw=1.2, label="Vrai")
    for name, (xt_, xu_, c, ls) in filter_results.items():
        ax.plot(t[WIN], xu_[WIN, i], color=c, lw=1.1, ls=ls, label=name)
    ax.set_ylabel(lbl); ax.legend(fontsize=8, ncol=5); ax.grid(True, ls="--", alpha=0.5)

axes[-1].set_xlabel("Pas de temps $k$ (100 premiers)")
fig.suptitle("Comparaison des filtres — modele Lotka-Volterra pairwise", y=1.01)
plt.tight_layout()
plt.show()

### 9.2 MSE par filtre et par composante

In [ ]:
filter_list = [
    ("EPKF",       xt_epkf[:M], xu_epkf[:M], "C0"),
    ("UPKF (wan)", xt_upkf[:M], xu_upkf[:M], "C1"),
    ("PPF",        xt_ppf[:M],  xu_ppf[:M],  "C2"),
    ("PF",         xt_pf[:M],   xu_pf[:M],   "C3"),
]

n_comp = 2
x_pos  = np.arange(n_comp)
w      = 0.18

fig, ax = plt.subplots(figsize=(8, 4))
for k, (name, xt_, xu_, c) in enumerate(filter_list):
    vals = [float(np.mean((xt_[:, i] - xu_[:, i]) ** 2)) for i in range(n_comp)]
    ax.bar(x_pos + (k - 1.5) * w, vals, w, label=name, color=c)

ax.set_xticks(x_pos)
ax.set_xticklabels([r"$x_1$ (proies)", r"$x_2$ (predateurs)"])
ax.set_ylabel("MSE")
ax.set_title("MSE par composante et par filtre")
ax.legend(fontsize=9); ax.grid(True, axis="y", ls="--", alpha=0.5)
plt.tight_layout()
plt.show()

---
## 10. Modèle augmenté

On instancie la version **augmentée** du modèle (dim_x=4, dim_y=2)
et on vérifie sa structure.

In [ ]:
model_aug = ModelFactoryNonLinear.create("model_x2_y2_LotkaVolterra_augmented")
params_aug = model_aug.get_params()

print(f"Modele augmente : {model_aug}")
print(f"dim_x : {params_aug['dim_x']}  (x1, x2, y1, y2)")
print(f"dim_y : {params_aug['dim_y']}  (y1, y2)")
print(f"augmented : {model_aug.augmented}")
print(f"pairwiseModel : {params_aug['pairwiseModel']}")
print()
print("mz0 augmente :", params_aug['mz0'].flatten())
print("mQ augmente (shape", params_aug['mQ'].shape, "):")
print(np.round(params_aug['mQ'], 4))

In [ ]:
latex_aug = model_aug.latex_model()
display(Math(latex_aug))

In [ ]:
# Filtrage avec le modele augmente (EPKF et PF — pas de PPF car non pairwise)
param_aug = make_param(model_aug)

epkf_aug = NonLinear_EPKF(param_aug, sKey=SEED)
res_aug   = epkf_aug.process_N_data(N=N)

# L etat estime est de dim 4 : on ne compare que les 2 premieres composantes (x1, x2)
xu_aug_full = np.array([r[4].flatten() for r in res_aug if r[4] is not None])
xt_aug_full = np.array([r[1].flatten() for r in res_aug[:len(xu_aug_full)]])

# Les 2 premieres composantes correspondent a x1, x2
xu_aug = xu_aug_full[:, :2]
xt_aug = xt_aug_full[:, :2]

print(f"EPKF (modele augmente, dim_x=4)  MSE x1,x2 = {mse(xt_aug, xu_aug):.6f}")

---
## 11. Récapitulatif

| Étape | Action |
|-------|--------|
| **1** | Définir `symbolic_model(sx, sy, st, su)` dans une classe héritant de `BaseModelGxGy` |
| **2** | Déposer le fichier `.py` dans `prg/models/nonLinear/` |
| **3** | La factory découvre automatiquement le modèle — aucune modification du code existant |
| **4** | Pour la version augmentée, hériter de `BaseModelFxHx` et réutiliser `_sgx`, `_sgy` du pairwise |
| **5** | Les jacobiens sont calculés automatiquement par SymPy dans les classes de base |

**Pour aller plus loin :**

| Tâche | Comment |
|-------|----------|
| Modifier les paramètres LV | Changer `ALPHA`, `BETA`, etc. en attributs de classe |
| Changer la fonction d'observation | Remplacer `x1 + C*y2` par une nonlinéarité, ex. `sp.log(x1)` |
| Tester d'autres sigma-points | `NonLinear_UPKF(param_lv, sigmaSet="cpkf")` |
| Augmenter le nombre de particules | `NonLinear_PPF(param_lv, n_particles=2000)` |
| Voir d'autres modèles pairwise | `model_x2_y2_pairwise`, `model_x2_y1_pairwise` |
